# 5.11 Case Study - House Price Prediction

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.11_case-study-house-price-prediction.ipynb)

Companion code for Section 5.11 of the book. Run the cells in order from top to bottom.

**Example 5.11.1**

In [ ]:
import numpy as np
df.rename(columns={'MedHouseVal': 'price'}, inplace=True)

print(f'\nDataset: {df.shape[0]:,} houses × {df.shape[1]} columns')
print(f'Target: Median house value (in $100,000s)')
print(f'Price range: ${df.price.min()*100:.0f}k - ${df.price.max()*100:.0f}k')
print(f'Missing values: {df.isnull().sum().sum()}')

# ── Step 2: EDA ───────────────────────────────────────────────────
print('\n--- Feature Statistics ---')
print(df.describe().round(3))

print('\n--- Target Distribution ---')
print(f'Mean:   ${df.price.mean()*100:.0f}k')
print(f'Median: ${df.price.median()*100:.0f}k')
print(f'Std:    ${df.price.std()*100:.0f}k')
print(f'Skew:   {stats.skew(df.price):+.4f}')

print('\n--- Correlation with Price ---')
corr_with_price = df.corr()['price'].drop('price').sort_values(key=abs, ascending=False)
print(corr_with_price.round(4))

# ── EDA Visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA - California Housing Dataset', fontsize=14, fontweight='bold')

# Target distribution
ax = axes[0,0]
ax.hist(df.price, bins=50, color='#1976D2', alpha=0.7, edgecolor='white')
ax.axvline(df.price.mean(), color='red', lw=2, label=f'Mean={df.price.mean():.2f}')
ax.set_xlabel('House Price ($100k)'); ax.set_title('Target Distribution')
ax.legend(); ax.grid(alpha=0.3)

# Income vs Price scatter
ax = axes[0,1]
sc = ax.scatter(df['MedInc'], df['price'], alpha=0.1, s=3, c=df['price'], cmap='viridis')
plt.colorbar(sc, ax=ax, label='Price')
ax.set_xlabel('Median Income'); ax.set_ylabel('Price ($100k)')
ax.set_title(f'Income vs Price (r={df.corr()["price"]["MedInc"]:.3f})')

# Geographic scatter (Lat/Long = California map proxy)
ax = axes[0,2]
sc2 = ax.scatter(df.Longitude, df.Latitude, c=df.price,
                 cmap='hot_r', alpha=0.5, s=2)
plt.colorbar(sc2, ax=ax, label='Price ($100k)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Geographic Price Distribution\n(California)')

# Correlation heatmap
ax = axes[1,0]
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr(), ax=ax, mask=mask, annot=True, fmt='.2f',
           cmap='RdYlGn', center=0, linewidths=0.3, annot_kws={'size':8})
ax.set_title('Correlation Matrix')

# Feature correlation with target
ax = axes[1,1]
colors = ['#1976D2' if v>0 else '#E53935' for v in corr_with_price]
ax.barh(corr_with_price.index, corr_with_price.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Pearson r'); ax.set_title('Feature-Price Correlation')

# Price by latitude (regional effect)
ax = axes[1,2]
df['lat_bin'] = pd.cut(df.Latitude, bins=10, labels=False)
df.groupby('lat_bin')['price'].mean().plot(kind='bar', ax=ax,
    color='#7B1FA2', edgecolor='white', alpha=0.8)
ax.set_xlabel('Latitude Bin (South→North)'); ax.set_ylabel('Mean Price ($100k)')
ax.set_title('Regional Price Variation (N→S)'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('case_study_eda.png', dpi=150, bbox_inches='tight')
plt.show()

**Example 5.11.2**

In [ ]:
# ── Step 3: Feature Engineering ──────────────────────────────────
df = df.drop('lat_bin', axis=1, errors='ignore')  # Drop temp column

print('Feature Engineering...')

# 1. Rooms and bedrooms per household (more meaningful than totals)
df['RoomsPerHousehold']   = df['AveRooms']
df['BedroomsPerRoom']     = df['AveBedrms'] / df['AveRooms'].clip(lower=0.01)

# 2. Population density
df['PopulationPerHousehold'] = df['Population'] / df['AveOccup'].clip(lower=0.01)

# 3. Income tiers (non-linear income effect)
df['IncomeSquared'] = df['MedInc']**2
df['IncomeSqrt']    = np.sqrt(df['MedInc'])

# 4. Geographic features (distance from major cities)
# LA: (34.05, -118.24)  SF: (37.77, -122.42)  San Diego: (32.72, -114.16)
for city, lat, lon in [('LA', 34.05, -118.24), ('SF', 37.77, -122.42),
                        ('SD', 32.72, -114.16)]:
    df[f'Dist_{city}'] = np.sqrt((df.Latitude-lat)**2 + (df.Longitude-lon)**2)

# 5. Log transform target (right-skewed)
df['log_price'] = np.log1p(df['price'])

print(f'Features after engineering: {df.shape[1]} columns')

# ── Step 4: Prepare X, y ──────────────────────────────────────────
feature_cols = [c for c in df.columns if c not in ['price','log_price']]
X = df[feature_cols]
y = df['price']         # Predict price directly
y_log = df['log_price'] # Alternative: predict log(price)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
_, _, yl_train, yl_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# ── Step 5: Model Comparison ───────────────────────────────────────
print('\nTraining models...')

# Configure all models
configs = {
    'OLS (baseline)':       LinearRegression(),
    'Ridge (α=1)':          Ridge(alpha=1.0),
    'Ridge (α=10)':         Ridge(alpha=10.0),
    'Lasso (α=0.01)':       Lasso(alpha=0.01, max_iter=10000),
    'ElasticNet':           ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
    'Ridge+Poly(deg=2)':    Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler2', StandardScaler()),
        ('ridge', Ridge(alpha=10.0))
    ]),
}

print(f'\n{"Model":<22} {"Train R²":>9} {"Test R²":>9} {"Test MAE":>10} {"Test RMSE":>11}')
print('='*65)
model_results = {}

for name, model in configs.items():
    if 'Poly' in name:
        model.fit(X_train, y_train)  # Poly pipeline has its own scaler
        ytr = model.predict(X_train)
        yte = model.predict(X_test)
    else:
        model.fit(X_tr, y_train)
        ytr = model.predict(X_tr)
        yte = model.predict(X_te)

    tr2   = r2_score(y_train, ytr)
    te2   = r2_score(y_test,  yte)
    te_mae = mean_absolute_error(y_test, yte)
    te_rmse= np.sqrt(mean_squared_error(y_test, yte))

    model_results[name] = {'train_r2':tr2,'test_r2':te2,
                            'test_mae':te_mae,'test_rmse':te_rmse,'model':model}
    print(f'{name:<22} {tr2:>9.4f} {te2:>9.4f} {te_mae:>10.4f} {te_rmse:>11.4f}')

# ── Step 6: Hyperparameter Tuning for Best Model ──────────────────
print('\nHyperparameter tuning (RidgeCV + PolynomialFeatures)...')

# Efficiently tune alpha via RidgeCV within polynomial pipeline
best_results = {}
for degree in [1, 2, 3]:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler2',StandardScaler()),
        ('ridge',  RidgeCV(alphas=np.logspace(-3,4,50), cv=5))
    ])
    pipe.fit(X_train, y_train)
    te2   = pipe.score(X_test, y_test)
    te_rmse = np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))
    opt_alpha = pipe.named_steps['ridge'].alpha_
    best_results[degree] = (te2, te_rmse, opt_alpha)
    print(f'  Poly(deg={degree})+RidgeCV: Test R²={te2:.4f}  RMSE={te_rmse:.4f}  alpha={opt_alpha:.4f}')

best_deg = max(best_results, key=lambda d: best_results[d][0])
print(f'\nBest degree: {best_deg}  (R²={best_results[best_deg][0]:.4f})')

**Example 5.11.3**

In [ ]:
# ── Step 7: Final Model ───────────────────────────────────────────
print('\n' + '='*65)
print('FINAL MODEL: Polynomial(degree=3) + RidgeCV')
print('='*65)

# Fit final model
final_model = Pipeline([
    ('poly',   PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge',  RidgeCV(alphas=np.logspace(-3,4,100), cv=5))
])
final_model.fit(X_train, y_train)

y_pred_final = final_model.predict(X_test)
y_pred_train = final_model.predict(X_train)

# All metrics
metrics = {
    'Train R²':          r2_score(y_train, y_pred_train),
    'Test R²':           r2_score(y_test, y_pred_final),
    'Train RMSE':        np.sqrt(mean_squared_error(y_train, y_pred_train)),
    'Test RMSE':         np.sqrt(mean_squared_error(y_test, y_pred_final)),
    'Test MAE':          mean_absolute_error(y_test, y_pred_final),
    'Test MAPE%':        np.mean(np.abs((y_test-y_pred_final)/y_test))*100,
    'Optimal alpha':     final_model.named_steps['ridge'].alpha_,
    'Poly features':     final_model.named_steps['poly'].n_output_features_,
}

print('\nFinal Model Performance:')
for k, v in metrics.items():
    unit = '%' if 'MAPE' in k else ''
    print(f'  {k:<20}: {v:.4f}{unit}')

# Business interpretation
residuals = y_test - y_pred_final
print(f'\nBusiness Interpretation:')
print(f'  Average prediction error: ${mean_absolute_error(y_test, y_pred_final)*100:.0f}k')
print(f'  Median prediction error:  ${np.median(np.abs(residuals))*100:.0f}k')
print(f'  Within $50k:  {(np.abs(residuals) < 0.5).mean():.1%} of predictions')
print(f'  Within $100k: {(np.abs(residuals) < 1.0).mean():.1%} of predictions')

# ── Step 8: Comprehensive Final Plots ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Case Study: House Price Prediction - Final Model Report',
            fontsize=14, fontweight='bold')

# 1. Actual vs Predicted
ax = axes[0,0]
ax.scatter(y_test, y_pred_final, alpha=0.4, s=8, c=y_test, cmap='viridis')
lims = [0, 5.5]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Price ($100k)'); ax.set_ylabel('Predicted Price ($100k)')
ax.set_title(f'Actual vs Predicted (R²={metrics["Test R²"]:.3f})')
ax.legend(); ax.grid(alpha=0.3)

# 2. Residuals vs Fitted
ax = axes[0,1]
ax.scatter(y_pred_final, residuals, alpha=0.4, s=6, color='#E53935')
ax.axhline(0, color='black', lw=2, linestyle='--')
ax.set_xlabel('Fitted Values (ŷ)'); ax.set_ylabel('Residuals (y-ŷ)')
ax.set_title('Residuals vs Fitted'); ax.grid(alpha=0.3)

# 3. Residual histogram
ax = axes[0,2]
ax.hist(residuals, bins=60, color='#7B1FA2', alpha=0.7, edgecolor='white', density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
       'r-', lw=2, label='Normal fit')
ax.set_xlabel('Residuals'); ax.set_title('Residual Distribution')
ax.legend(); ax.grid(alpha=0.3)

# 4. Q-Q plot
ax = axes[1,0]
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Q-Q Plot')

# 5. Model comparison bar chart
ax = axes[1,1]
model_names = list(model_results.keys()) + ['Final Model\n(Poly3+Ridge)']
test_r2s    = [model_results[m]['test_r2'] for m in model_results] + [metrics['Test R²']]
colors_bar  = ['#BDBDBD']*len(model_results) + ['#1976D2']
bars = ax.bar(range(len(model_names)), test_r2s, color=colors_bar, edgecolor='white')
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Test R²'); ax.set_title('Model Comparison'); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0.55, 0.78)
for bar, r2 in zip(bars, test_r2s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
           f'{r2:.3f}', ha='center', fontsize=8, fontweight='bold')

# 6. Prediction error distribution
ax = axes[1,2]
abs_errors = np.abs(residuals) * 100  # Convert to $k
ax.hist(abs_errors, bins=50, color='#FF6F00', alpha=0.7, edgecolor='white', cumulative=True,
       density=True, histtype='stepfilled', label='CDF of |error|')
for threshold, label in [(50,'$50k'),(100,'$100k'),(200,'$200k')]:
    pct = (abs_errors < threshold).mean()
    ax.axvline(threshold, color='red', linestyle='--', alpha=0.7)
    ax.text(threshold+5, pct-0.05, f'{pct:.0%}\n<{label}', fontsize=9)
ax.set_xlabel('Absolute Error ($k)'); ax.set_ylabel('Cumulative Fraction')
ax.set_title('Error Distribution\n(Cumulative)')
ax.set_xlim(0, 400); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('case_study_final_report.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*65)
print('CASE STUDY COMPLETE')
print('='*65)
print(f'Baseline OLS R²:     {model_results["OLS (baseline)"]["test_r2"]:.4f}')
print(f'Final Model R²:      {metrics["Test R²"]:.4f}')
print(f'Improvement:         +{(metrics["Test R²"]-model_results["OLS (baseline)"]["test_r2"])*100:.1f}% R²')
print(f'Average prediction is within ${metrics["Test MAE"]*100:.0f}k of true price')